<h1 style="font-family: Georgia, serif; text-align:center; font-weight:400; line-height: 1.2;">
    <br>
    <span style="font-size: 60px;">Bernanke-Blanchard Model</span><br>
    <span style="font-size: 30px;">Data Processing & Variable Construction</span>
</h1>

<br>

<p style="text-align:center; font-size:18px; margin-top: 15px; font-family: Georgia, serif;">
    <span style="font-size: 30px;">Diploma Thesis</span><br>
    <b style="font-size: 16px; ">Response of Inflation to Price Shocks: Does the Price Level Matter?</b><br>

<hr style="height:3px; background:black; width:100%; margin: 20px auto;">

<p style="text-align:center; font-size:20px; font-weight:600; margin-bottom: 4px; font-family: Georgia, serif;">
    Václav Šmíro
</p>

<p style="text-align:center; font-size:15px; color:#444; font-family: Georgia, serif;">
    vaclavsmiro@proton.me
</p>

In [101]:
import pandas as pd
import numpy as np
import os

# 1) Endogenous variables
<hr style="height:3px; background:black; width:100%; margin: 20px auto;">

## 1.1) Wage growth

| Name | Description | Transformation |
|:---|:---|:---|
| **gw** | Total economy, hourly compensation (Code: *MNA.Q.Y.CZ.W2.S1.S1._Z.COM_HW._Z._T._Z.IX.V.N*). | **Growth:** Annualized QoQ growth rate ($400 \cdot \ln(W_t/W_{t-1})$). |

**Source:** [ECB Data Portal](https://data.ecb.europa.eu/data/datasets/MNA/MNA.Q.Y.CZ.W2.S1.S1._Z.COM_HW._Z._T._Z.IX.V.N)

In [102]:
# 1. Load the dataset
df_wage = pd.read_csv("input_data/hourly_compensation.csv")

# 2. Select and rename columns to model variables
target_col = "Hourly compensation (MNA.Q.Y.CZ.W2.S1.S1._Z.COM_HW._Z._T._Z.IX.V.N)"
df_wage = df_wage[["TIME PERIOD", target_col]]

df_wage = df_wage.rename(columns={
    "TIME PERIOD": "date",
    target_col: "wage"
})

# 3.  Set the date as index
df_wage["date"] = pd.PeriodIndex(df_wage["date"], freq="Q")
df_wage = df_wage.set_index("date").sort_index()

#4. YoY% annualized (QoQ% is volatile)
df_wage['gw'] = 400 * np.log(df_wage['wage'] / df_wage['wage'].shift(1))

# 5. Select the columns
df_wage = df_wage[["gw", "wage"]]

# 6. The transformed dataset
print(df_wage.head())

               gw   wage
date                    
1995Q1        NaN  21.46
1995Q2   2.230489  21.58
1995Q3  24.968823  22.97
1995Q4  18.212037  24.04
1996Q1  13.738063  24.88


## 1.2) Price growth – Overall HICP and CPI

| Name | Description | Transformation |
|:---|:---|:---|
| **ghicp** | Harmonised Index of Consumer Prices (Code: *ICP.M.CZ.N.000000.4.ANR*). | **Growth:** Seasonally adjusted (JDemetra+), aggregated to quarterly mean, then annualized QoQ growth ($400 \cdot \ln(P_t/P_{t-1})$). |
| **gcpi** | National Consumer Price Index. Merged series combining pre- and post-2018 data to ensure continuity despite ECOICOP reclassification. | **Growth:** Seasonally adjusted via TRAMO/SEATS (JDemetra+), aggregated to quarterly mean, then annualized QoQ growth ($400 \cdot \ln(P_t/P_{t-1})$). |

**Sources:** [ECB Data Portal](https://data.ecb.europa.eu/data/datasets/ICP/ICP.M.CZ.N.000000.4.ANR) (HICP); [CZSO (2018+)](https://vdb.czso.cz/vdbvo2/faces/cs/index.jsf?page=vystup-objekt&pvo=CEN083A&z=T&f=TABULKA&skupId=2218&katalog=31779&pvo=CEN083A&evo=v2504_!_CEN-SPO-BAZIC2015-EM_1) and [CZSO (pre-2018)](https://vdb.czso.cz/vdbvo2/faces/cs/index.jsf?page=vystup-objekt&pvo=CEN080&z=T&f=TABULKA&skupId=43&katalog=31779&pvo=CEN080&evo=v2300_!_CEN-SPO-BAZIC2005-M_1).

*Note: The CPI series is constructed by merging two datasets due to the 2018 transition to the ECOICOP classification system. While the item breakdown changed, the index calculation methodology remained consistent, allowing for a continuous time series.*

In [103]:
# 1. Load the pre-processed dataset
# Note: This input file is NOT raw data. It contains the time series that has already been:
# - Spliced to combine pre- and post-2018 data (bridging the ECOICOP classification change).
# - Seasonally adjusted using TRAMO/SEATS in JDemetra+.
df_price = pd.read_csv("input_data/headline_inflation.csv")
df_price["date"] = pd.to_datetime(df_price["date"], format="%m/%d/%y")
df_price = df_price.sort_values("date").set_index("date")
df_price = df_price.resample("QE").mean()

# 3. Calculate annualized QoQ growth rate and rename variables
df_price['ghicp'] = 400 * np.log(df_price['hicp_headline_sa'] / df_price['hicp_headline_sa'].shift(1))
df_price['gcpi'] = 400 * np.log(df_price['cpi_headline_sa'] / df_price['cpi_headline_sa'].shift(1))

# 4. Convert date index to Quarterly period format
df_price.index = df_price.index.to_period("Q")

# 5. Select the renamed columns and save
df_price = df_price[['ghicp', 'gcpi']]

print(df_price.head())

           ghicp      gcpi
date                      
1996Q1       NaN       NaN
1996Q2  8.462830  7.497090
1996Q3  6.625390  7.511437
1996Q4  6.751365  7.339409
1997Q1  4.921751  5.404274


## 1.3) Inflation Expectation

| Name | Description | Transformation |
|:---|:---|:---|
| **cf1** | 1-year-ahead inflation expectations of managers of non-financial corporations (Firms). Code: *SINFLNFKQ1*. | ––– |
| **cf3** | 3-year-ahead financial market inflation expectations. Code: *SINFLFTM2*. |  ––– |

**Sources:** [CNB ARAD Database](https://www.cnb.cz/arad/#/en/display_link/basket__SINFLFTM2.SINFLNFKQ1_)

*Note: Short-term expectations use data from non-financial corporations as they better reflect the real economy's price-setting behavior. Long-term expectations (the anchor) rely on financial analysts' data, as they provide a consistent long-term time series and better represent the credibility of the inflation target in financial markets.*

In [104]:
# 1. Load the dataset
df_expe = pd.read_csv("input_data/inflation_expectation.csv", sep=";")

# 2. Rename columns to model variables
df_expe = df_expe.rename(columns={
    "Period": "date",
    "SINFLFTM2 * (%)": "cf3",
    "SINFLNFKQ1 * (%)": "cf1"
})

# 3. Convert the 'date' column to datetime objects, transform date to Quarterly period format and sort the date
df_expe["date"] = pd.to_datetime(df_expe["date"])
df_expe["date"] = df_expe["date"].dt.to_period("Q")
df_expe = df_expe.sort_values("date").set_index("date")

# 4. Select the columns
df_expe = df_expe[["cf1", "cf3"]]

# 5. The transformed dataset
print(df_expe.head())

        cf1  cf3
date            
1999Q4  3.9  4.1
2000Q1  4.3  4.2
2000Q2  4.8  3.9
2000Q3  5.0  4.1
2000Q4  4.7  4.0


## 1.4) Catch-up term

| Name | Description | Transformation |
|:---|:---|:---|
| **diff_hicp** | Realized HICP inflation surprise (Catch-up). Measures the gap between the past year's inflation and previous expectations. | ––– |
| **diff_cpi** | Realized CPI inflation surprise (Catch-up). Measures the gap between the past year's inflation and previous expectations. | ––– |

**Sources:** Derived from [CNB ARAD Database](https://www.cnb.cz/arad/#/en/display_link/basket__SINFLFTM2.SINFLNFKQ1_) (expectations); [ECB Data Portal](https://data.ecb.europa.eu/data/datasets/ICP/ICP.M.CZ.N.000000.4.ANR) (HICP); [CZSO (2018+)](https://vdb.czso.cz/vdbvo2/faces/cs/index.jsf?page=vystup-objekt&pvo=CEN083A&z=T&f=TABULKA&skupId=2218&katalog=31779&pvo=CEN083A&evo=v2504_!_CEN-SPO-BAZIC2015-EM_1) and [CZSO (pre-2018)](https://vdb.czso.cz/vdbvo2/faces/cs/index.jsf?page=vystup-objekt&pvo=CEN080&z=T&f=TABULKA&skupId=43&katalog=31779&pvo=CEN080&evo=v2300_!_CEN-SPO-BAZIC2005-M_1)


In [105]:
# 1. Join the existing dataframes (from inflation expectation syntax and price growth syntax)
df_catch = df_price.join(df_expe[['cf1']])

# 2. Calculate rolling 4-quarter average of inflation
df_catch['hicp_4q_avg'] = df_catch['ghicp'].rolling(window=4).mean()
df_catch['cpi_4q_avg'] = df_catch['gcpi'].rolling(window=4).mean()

# 3. Shift 1-year expectations (cf1) by 4 quarters
df_catch['cf1_lagged'] = df_catch['cf1'].shift(4)

# 4. Calculate the Catch-up terms
df_catch['diff_hicp'] = df_catch['hicp_4q_avg'] - df_catch['cf1_lagged']
df_catch['diff_cpi'] = df_catch['cpi_4q_avg'] - df_catch['cf1_lagged']

# 5. Select the columns
df_catch = df_catch[["diff_hicp", "diff_cpi"]]

# 6. The transformed dataset (limited by cf1, we have data from 2000Q4)
print(df_catch.head(20))

        diff_hicp  diff_cpi
date                       
1996Q1        NaN       NaN
1996Q2        NaN       NaN
1996Q3        NaN       NaN
1996Q4        NaN       NaN
1997Q1        NaN       NaN
1997Q2        NaN       NaN
1997Q3        NaN       NaN
1997Q4        NaN       NaN
1998Q1        NaN       NaN
1998Q2        NaN       NaN
1998Q3        NaN       NaN
1998Q4        NaN       NaN
1999Q1        NaN       NaN
1999Q2        NaN       NaN
1999Q3        NaN       NaN
1999Q4        NaN       NaN
2000Q1        NaN       NaN
2000Q2        NaN       NaN
2000Q3        NaN       NaN
2000Q4   0.314429  0.066384


# 2) Exogenous variables
<hr style="height:3px; background:black; width:100%; margin: 20px auto;">

## 2.1) Energy & Food price growth (HICP & CPI)

| Name | Description | Transformation |
|:---|:---|:---|
| **grpe_hicp** | Growth of the relative price of energy (HICP). Code: *ICP.M.CZ.N.NRGY00.4.INX*. | **Relative Growth:** Annualized QoQ growth of the ratio between the HICP Energy index and Hourly Compensation ($400 \cdot \Delta \ln(P_{energy}/W)$). |
| **grpf_hicp** | Growth of the relative price of food (HICP). Code: *ICP.M.CZ.N.011000.4.INX*. | **Relative Growth:** Annualized QoQ growth of the ratio between the HICP Food index and Hourly Compensation ($400 \cdot \Delta \ln(P_{food}/W)$). |
| **grpe_cpi** | Growth of the relative price of energy, CZ-COICOP-[01] | **Relative Growth:** Annualized QoQ growth of the ratio between the National CPI Energy index and Hourly Compensation ($400 \cdot \Delta \ln(P_{energy}/W)$). |
| **grpf_cpi** | Growth of the relative price of food, CZ-COICOP-[04] | **Relative Growth:** Annualized QoQ growth of the ratio between the National CPI Food index and Hourly Compensation ($400 \cdot \Delta \ln(P_{food}/W)$). |

**Sources:** ECB Data Portal: [HICP Food](https://data.ecb.europa.eu/data/datasets/ICP/ICP.M.CZ.N.011000.4.INX), [HICP Energy](https://data.ecb.europa.eu/data/datasets/ICP/ICP.M.CZ.N.NRGY00.4.INX), [Wage](https://data.ecb.europa.eu/data/datasets/MNA/MNA.Q.Y.CZ.W2.S1.S1._Z.COM_HW._Z._T._Z.IX.V.N); CZSO: [CPI Food and Energy prices](https://vdb.czso.cz/vdbvo2/faces/cs/index.jsf?page=vystup-objekt&pvo=CEN083A)

### 2.1.1) **CPI** – Energy & Food price growth

In [106]:
# 1. Load the first dataset (1995 - 2017)
df_part1 = pd.read_excel("input_data/cpi_1995.xlsx", sheet_name="DATA", skiprows=7, header=None, usecols=[1, 3, 6])
df_part1.columns = ["date", "cpi_food", "cpi_energy"]

# 2. Load the second dataset (2018 - present)
df_part2 = pd.read_excel("input_data/cpi_2018.xlsx", sheet_name="DATA", skiprows=7, header=None, usecols=[1, 3, 6])
df_part2.columns = ["date", "cpi_food", "cpi_energy"]

# 3. Concatenate the datasets and setting the date as index (quartely)
df_cpi = pd.concat([df_part1, df_part2])
df_cpi = df_cpi.dropna()
df_cpi["date"] = pd.to_datetime(df_cpi["date"], format = "%m/%Y")
df_cpi = df_cpi.sort_values("date").set_index("date")
df_cpi = df_cpi.resample("QE").mean()
df_cpi.index = df_cpi.index.to_period("Q")

# 4. The transformed dataset
print(df_cpi.head())

         cpi_food cpi_energy
date                        
1995Q1  64.666667  24.933333
1995Q2  65.500000  25.333333
1995Q3  63.933333  26.866667
1995Q4  65.933333  27.366667
1996Q1  68.733333  27.833333


/Users/smirovaclav/myenv/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/smirovaclav/myenv/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


### 2.1.2) **HCPI** – Energy & Food price growth

In [107]:
# 1. Load the datasets
df_food = pd.read_csv("input_data/hicp_food.csv")
df_energy = pd.read_csv("input_data/hicp_energy.csv")

# 2. Rename columns to model variables
# Food
col_food = "HICP - Food (ICP.M.CZ.N.011000.4.INX)"
df_food = df_food.rename(columns={"DATE": "date", col_food: "hicp_food"})
df_food = df_food[["date", "hicp_food"]]

# Energy
col_energy = "HICP - Energy (ICP.M.CZ.N.NRGY00.4.INX)"
df_energy = df_energy.rename(columns={"DATE": "date", col_energy: "hicp_energy"})
df_energy = df_energy[["date", "hicp_energy"]]

# 3. Merge datasets
df_hicp = pd.merge(df_food, df_energy, on="date", how="outer")

# 4. Set the date and Index
df_hicp["date"] = pd.to_datetime(df_hicp["date"], format="%Y-%m-%d")
df_hicp = df_hicp.sort_values("date").set_index("date")
df_hicp = df_hicp.resample("QE").mean()
df_hicp.index = df_hicp.index.to_period("Q")

# 5. The transformed dataset
print(df_hicp.head())

        hicp_food  hicp_energy
date                          
1999Q4  70.300000    51.300000
2000Q1  70.866667    54.600000
2000Q2  70.333333    56.066667
2000Q3  70.933333    57.100000
2000Q4  72.733333    57.033333


### 2.1.3) **Merge** – Price shocks relative to wage

In [109]:
# 1. Join shocks with the wage
df_shocks = df_hicp.join([df_wage[['wage']], df_cpi], how='inner')
df_shocks = df_shocks.apply(pd.to_numeric, errors='coerce')

# A) Energy vs. Wages (HICP)
df_shocks['grpe_hicp'] = 400 * (
    np.log(df_shocks["hicp_energy"] / df_shocks["wage"]) - 
    np.log(df_shocks["hicp_energy"].shift(1) / df_shocks["wage"].shift(1))
)

# B) Food vs. Wages (HICP)
df_shocks['grpf_hicp'] = 400 * (
    np.log(df_shocks["hicp_food"] / df_shocks["wage"]) - 
    np.log(df_shocks["hicp_food"].shift(1) / df_shocks["wage"].shift(1))
)

# C) Energy vs. Wages (CPI)
df_shocks['grpe_cpi'] = 400 * (
    np.log(df_shocks["cpi_energy"] / df_shocks["wage"]) - 
    np.log(df_shocks["cpi_energy"].shift(1) / df_shocks["wage"].shift(1))
)

# D) Food vs. Wages (CPI)
df_shocks['grpf_cpi'] = 400 * (
    np.log(df_shocks["cpi_food"] / df_shocks["wage"]) - 
    np.log(df_shocks["cpi_food"].shift(1) / df_shocks["wage"].shift(1))
)

# 2. Select the columns
df_shocks = df_shocks[["grpe_hicp", "grpf_hicp", "grpe_cpi", "grpf_cpi"]]

# 3. The transformed dataset
print(df_shocks.head())

        grpe_hicp  grpf_hicp   grpe_cpi  grpf_cpi
date                                             
1999Q4        NaN        NaN        NaN       NaN
2000Q1  20.067627  -1.658274  15.410624  1.604504
2000Q2  11.526019  -2.098744   3.169484 -1.838818
2000Q3  -3.295358  -7.202571  -3.935657 -7.471811
2000Q4  -7.049505   3.441511  -4.385078  2.659419


## 2.2) Labor market tightness

### 2.2.1) Vaccancy to unemployed ratio

| Name | Description | Transformation |
|:---|:---|:---|
| **vu** | Ratio of job vacancies to unplaced job seekers (MoLSA data). Codes: *SNEZAMM41* (Vacancies) and *SNEZAMM31* (Seekers). | **Ratio:** Monthly data aggregated to quarterly means, then calculated as $V_t / U_t$. |

**Sources:** [CNB ARAD Database](https://www.cnb.cz/arad/#/en/display_link/basket__SNEZAMM41.SNEZAMM31_)

In [110]:
# 1. Load the dataset
df_vu = pd.read_csv("input_data/vu_ratio.csv", sep=";")

# 2. Rename columns to model variables
df_vu = df_vu.rename(columns={
    "Period": "date",
    "SNEZAMM41 * (Number)": "vacancies",
    "SNEZAMM31 * (Number)": "unemployment"
})

# 3. Convert the 'date' column to datetime objects, transform date to Quarterly period format and sort the date
df_vu["date"] = pd.to_datetime(df_vu["date"])
df_vu["date"] = df_vu["date"].dt.to_period("Q")
df_vu = df_vu.sort_values("date").set_index("date")

# 4. Calculating the v/u ratio
df_vu['vu'] = (df_vu["vacancies"] / df_vu["unemployment"])

# 5. Select the columns
df_vu = df_vu[["vu"]]

# 6. The transformed dataset
print(df_vu.head())

              vu
date            
1993Q4  0.291037
1994Q1  0.359350
1994Q2  0.490319
1994Q3  0.501525
1994Q4  0.460060


### 2.2.2) Unemployment rate 

| Name | Description | Transformation |
|:---|:---|:---|
| **u_gap** | Unemployment rate (age 15 to 74), Code: *LFSI.M.CZ.S.UNEHRT.TOTAL0.15_74.M*. | **Deviation from Mean:** Monthly SA rate aggregated to quarterly mean, calculated as the deviation from the sample average ($u_t - \bar{u}$). |

**Source:** [ECB Data Portal](https://data.ecb.europa.eu/data/datasets/LFSI/LFSI.M.CZ.S.UNEHRT.TOTAL0.15_74.M)

*Note: Negative values indicate a tight labor market (overheating), while positive values indicate slack.*

In [111]:
# 1. Load the dataset
df_u = pd.read_csv("input_data/unemployment_rate.csv")

# 2. Select and rename columns to model variables
target_col = "Unemployment rate, age 15 to 74, male (LFSI.M.CZ.S.UNEHRT.TOTAL0.15_74.M)"
df_u = df_u[["DATE", target_col]]

df_u = df_u.rename(columns={
    "DATE": "date",
    target_col: "u_rate"
})

# 3.  Set the date to period
df_u["date"] = pd.to_datetime(df_u["date"], format="%m/%d/%y")
df_u = df_u.sort_values("date").set_index("date")

df_u["u_rate"] = pd.to_numeric(df_u["u_rate"], errors='coerce')
df_u = df_u.resample("QE").mean()

df_u.index = df_u.index.to_period("Q")


# 4. Calculatiing the unemployment gap
# unemployment rate (v %)
u = df_u["u_rate"]
# sample mean
u_bar = u.mean()
df_u["u_gap"] = u - u_bar

# 5. Select the columns
df_u_gap = df_u[["u_gap"]]

# 6. The transformed dataset
print(df_u_gap.head())

           u_gap
date            
1993Q1 -0.799621
1993Q2 -0.966288
1993Q3 -1.066288
1993Q4 -1.099621
1994Q1 -1.132955


## 2.3) Shortage - GSCPI Index

| Name | Description | Transformation |
|:---|:---|:---|
| **shortage** | Global Supply Chain Pressure Index (disruption measure). | **Standardized:** Quarterly mean scaled by 1999–2019 baseline $\mu$ and $\sigma$. |

**Source:** [NYFED](https://www.newyorkfed.org/research/policy/gscpi#/overview)

In [112]:
# 1. Load the Excel file (skiprows=5 ignores the logo and empty rows)
df_gscpi = pd.read_excel("input_data/gscpi_data.xls", sheet_name="GSCPI Monthly Data", skiprows=5, header=None)

# 2. Rename columns
# Column A (0) is Date, Column B (1) is the GSCPI index value
df_gscpi = df_gscpi.rename(columns={0: "date", 1: "gscpi"})

# 3. Date conversion
df_gscpi["date"] = pd.to_datetime(df_gscpi["date"], format="%d-%b-%Y")
df_gscpi = df_gscpi.sort_values("date").set_index("date")
df_gscpi = df_gscpi.resample("QE").mean()

# 4. Calculate standardization parameters from the baseline period
# Baseline is defined as 1998 to 2019 to represent "normal" supply chain conditions
baseline = df_gscpi.loc['1998-03-31':'2019-12-31']
mean_normal = baseline['gscpi'].mean()
std_normal = baseline['gscpi'].std()

# 5. Standardize the entire dataset
df_gscpi['shortage'] = (df_gscpi['gscpi'] - mean_normal) / std_normal

# 6. Date to q. period
df_gscpi.index = df_gscpi.index.to_period("Q")

# 7. Select the columns
df_gscpi = df_gscpi[['shortage']]

# 8. The transformed dataset
print(df_gscpi.head())

        shortage
date            
1998Q1 -0.579162
1998Q2 -0.358731
1998Q3 -1.429482
1998Q4 -1.005833
1999Q1  0.198089


## 2.4) Produktivity trend

| Name | Description | Transformation |
|:---|:---|:---|
| **gpty** | Total economy labour productivity per hour worked, Code: *MNA.Q.Y.CZ.W0.S1.S1._Z.LPR_HW._Z._T._Z.IX.LR.N* | **Trend:** Annualized QoQ growth ($400 \cdot \ln(P_t/P_{t-1})$) smoothed with an 8-quarter moving average. |

**Source:** [ECB Data Portal](https://data.ecb.europa.eu/data/datasets/MNA/MNA.Q.Y.CZ.W0.S1.S1._Z.LPR_HW._Z._T._Z.IX.LR.N)

In [114]:
# 1. Load the dataset
df_productivty = pd.read_csv("input_data/productivity_per_hour.csv")

# 2. Select and rename columns to model variables
target_col = "Labour Productivity (per hours worked) (MNA.Q.Y.CZ.W0.S1.S1._Z.LPR_HW._Z._T._Z.IX.LR.N)"
df_productivty = df_productivty[["TIME PERIOD", target_col]]

df_productivty = df_productivty.rename(columns={
    "TIME PERIOD": "date",
    target_col: "productivity"
})

# 3.  Set the date as index
df_productivty["date"] = pd.PeriodIndex(df_productivty["date"], freq="Q")
df_productivty = df_productivty.set_index("date").sort_index()

#4. QoQ% annualized
df_productivty['gp'] = 400 * np.log(df_productivty['productivity'] / df_productivty['productivity'].shift(1))

# 5. 8-quarter MA
df_productivty['gpty'] = df_productivty['gp'].rolling(window=8).mean()

# 6. Select the columns
df_productivty = df_productivty[["gpty"]]

# 7. The transformed dataset
print(df_productivty.head(10))

            gpty
date            
1995Q1       NaN
1995Q2       NaN
1995Q3       NaN
1995Q4       NaN
1996Q1       NaN
1996Q2       NaN
1996Q3       NaN
1996Q4       NaN
1997Q1  2.807236
1997Q2  2.172681


# 3) Merge
<hr style="height:3px; background:black; width:100%; margin: 20px auto;">

In [115]:
# 1. Create a list of all DataFrames 
dfs_to_merge = [
    df_wage[["gw"]], 
    df_price[['ghicp', 'gcpi']], 
    df_expe[["cf1", "cf3"]], 
    df_catch[["diff_hicp", "diff_cpi"]], 
    df_shocks[["grpe_hicp", "grpf_hicp", "grpe_cpi", "grpf_cpi"]], 
    df_vu[["vu"]], 
    df_u_gap[["u_gap"]], 
    df_gscpi[['shortage']], 
    df_productivty[["gpty"]]
]

# 2. Merge and Cleaning
df_final = pd.concat(dfs_to_merge, axis=1, join='inner')
df_final = df_final.sort_index().dropna()

# 3. New folder creation
folder_name = "final_dataset"
if not os.path.exists(folder_name):
    os.makedirs(folder_name)

# 4. Save to CSV
df_final.to_csv(f"{folder_name}/intermediate_data_per_hour_cze.csv")
df_final.head()

,gw,ghicp,gcpi,cf1,cf3,diff_hicp,diff_cpi,grpe_hicp,grpf_hicp,grpe_cpi,grpf_cpi,vu,u_gap,shortage,gpty
date,,,,,,,,,,,,,,,
2000Q4,6.582215,3.863274,4.214488,4.7,4.0,0.314429,0.066384,-7.049505,3.441511,-4.385078,2.659419,0.113905,2.533712,-1.001645,3.147008
2001Q1,22.272796,5.185574,5.146869,4.2,3.4,-0.317527,-0.334728,1.330193,-14.287050,1.590198,-15.875503,0.123367,2.333712,-1.072582,5.434594
2001Q2,15.096131,5.168087,5.402637,4.8,3.8,0.061352,0.023109,-7.239905,-6.386611,-12.581119,-5.860260,0.142279,2.367045,-1.458087,5.965224
2001Q3,7.191205,5.422794,6.295594,4.7,3.6,-0.090068,0.264897,4.101156,-7.367068,4.930935,-8.398883,0.142792,2.467045,-1.063739,4.774160
2001Q4,15.817707,0.035430,-0.192080,3.9,3.6,-0.747029,-0.536745,-23.239070,-18.996531,-14.493202,-18.765910,0.112795,2.167045,-2.192281,5.947117
